# Adam y comparación controlada con SGD

**Capítulo 2 · Universidad de las Hespérides**

Adaptación al español de *Dive into Deep Learning*, Aston Zhang, Zachary C. Lipton, Mu Li y Alexander J. Smola.
Fuente: `locked/chapter_optimization/adam.ipynb` · [Lección original](https://d2l.ai/chapter_optimization/adam.html).
Texto adaptado bajo [CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/). [Procedencia y cambios](../PROCEDENCIA.md).
Se conserva la secuencia de las celdas y de los ejercicios; las notas de Hespérides se identifican expresamente.

**Entorno:** ejecuta `uv sync` en la raíz y selecciona su Python como kernel. Las descargas se realizan una vez y quedan en `data/`.
Por defecto, el soporte limita los entrenamientos de `Trainer` a tres épocas y 1024/256 ejemplos para CPU.
Para repetir el régimen completo, inicia Jupyter con `HESPERIDES_COMPLETO=1`. Los ejemplos visuales pequeños conservan su propia configuración explícita.
Los datos de texto en inglés o francés son entradas de los experimentos originales y mantienen su idioma.


In [ ]:
from pathlib import Path
import sys
RAIZ = Path.cwd() if (Path.cwd() / "laboratorio").exists() else Path.cwd().parent
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))
from laboratorio import d2l, configurar, epocas
configurar()


# Adam
<a id="sec_adam"></a>

En las discusiones previas a esta sección nos encontramos con una serie de técnicas para la optimización eficiente. Vamos a recapitularlas en detalle aquí:

* Vimos que [Referencia sec_sgd](https://d2l.ai/chapter_optimization/sgd.html#sec-sgd) es más eficaz que Gradient Descent al resolver problemas de optimización, por ejemplo, debido a su resistencia inherente a datos redundantes.
* Vimos que [Referencia sec_minibatch_sgd](https://d2l.ai/chapter_optimization/minibatch-sgd.html#sec-minibatch-sgd) ofrece eficiencia adicional significativa derivada de la vectorización, utilizando conjuntos más grandes de observaciones en un minibatch. Esta es la clave para el procesamiento eficiente multi-máquina, multi-GPU y paralelo general.
* [Referencia sec_momentum](https://d2l.ai/chapter_optimization/momentum.html#sec-momentum) agregó un mecanismo para agregar una historia de gradientes pasados para acelerar la convergencia.
* [Referencia sec_adagrad](https://d2l.ai/chapter_optimization/adagrad.html#sec-adagrad) utilizado escalado por coordenadas para permitir un precondicionador computacionalmente eficiente.
* Escalado de [Referencia sec_rmsprop](https://d2l.ai/chapter_optimization/rmsprop.html#sec-rmsprop) disociado por coordenadas de un ajuste de la tasa de aprendizaje.

Adam [Kingma.Ba.2014](https://d2l.ai/chapter_references/zreferences.html) combina todas estas técnicas en un algoritmo de aprendizaje eficiente. Como era de esperar, este es un algoritmo que se ha vuelto bastante popular como uno de los algoritmos de optimización más robustos y eficaces para usar en el aprendizaje profundo. No es sin problemas, sin embargo. En particular, [Reddi.Kale.Kumar.2019](https://d2l.ai/chapter_references/zreferences.html) muestran que hay situaciones en las que Adam puede diverger debido a un control de varianza pobre. En un trabajo de seguimiento [Zaheer.Reddi.Sachan.ea.2018](https://d2l.ai/chapter_references/zreferences.html) propuso un hotfix a Adam, llamado Yogi que aborda estos problemas. Más sobre esto más adelante. Por ahora vamos a revisar el algoritmo de Adam.

## El Algoritmo
Uno de los componentes clave de Adam es que utiliza promedios móviles ponderados exponenciales (también conocido como promedio de fugas) para obtener una estimación tanto del momento como del segundo momento del gradiente. Es decir, utiliza las variables de estado

$$\begin{aligned}
    \mathbf{v}_t & \leftarrow \beta_1 \mathbf{v}_{t-1} + (1 - \beta_1) \mathbf{g}_t, \\
    \mathbf{s}_t & \leftarrow \beta_2 \mathbf{s}_{t-1} + (1 - \beta_2) \mathbf{g}_t^2.
\end{aligned}$$

Aquí $\beta_1$ y $\beta_2$ son parámetros de ponderación no negativos. Las opciones comunes para ellos son $\beta_1 = 0.9$ y $\beta_2 = 0.999$. Es decir, la estimación de varianza se mueve *mucho más lentamente* que el término de impulso. Tenga en cuenta que si inicializamos $\mathbf{v}_0 = \mathbf{s}_0 = 0$ tenemos una cantidad significativa de sesgo inicialmente hacia valores más pequeños. Esto se puede abordar usando el hecho de que $\sum_{i=0}^{t-1} \beta^i = \frac{1 - \beta^t}{1 - \beta}$ para volver a normalizar términos. Correspondientemente las variables de estado normalizado son dadas por

$$\hat{\mathbf{v}}_t = \frac{\mathbf{v}_t}{1 - \beta_1^t} \textrm{ and } \hat{\mathbf{s}}_t = \frac{\mathbf{s}_t}{1 - \beta_2^t}.$$

Armados con las estimaciones adecuadas ahora podemos escribir las ecuaciones de actualización. Primero, reescalamos el gradiente de una manera muy similar a la de RMSProp para obtener

$$\mathbf{g}_t' = \frac{\eta \hat{\mathbf{v}}_t}{\sqrt{\hat{\mathbf{s}}_t} + \epsilon}.$$

A diferencia de RMSProp nuestra actualización utiliza el impulso $\hat{\mathbf{v}}_t$ en lugar del gradiente en sí. Además, hay una ligera diferencia cosmética como el escalado ocurre utilizando $\frac{1}{\sqrt{\hat{\mathbf{s}}_t} + \epsilon}$ en lugar de $\frac{1}{\sqrt{\hat{\mathbf{s}}_t + \epsilon}}$. El primero trabaja posiblemente ligeramente mejor en la práctica, de ahí la desviación de RMSProp. Típicamente elegimos $\epsilon = 10^{-6}$ para una buena compensación entre estabilidad numérica y fidelidad.

Ahora tenemos todas las piezas en su lugar para calcular actualizaciones. Esto es ligeramente anticlimática y tenemos una simple actualización de la forma

$$\mathbf{x}_t \leftarrow \mathbf{x}_{t-1} - \mathbf{g}_t'.$$

Revisar el diseño de Adam su inspiración es claro. Momentum y escala son claramente visibles en las variables de estado. Su definición bastante peculiar nos obliga a debias términos (esto podría ser fijado por una condición ligeramente diferente de inicialización y actualización). En segundo lugar, la combinación de ambos términos es bastante simple, dado RMSProp. Por último, la tasa de aprendizaje explícita $\eta$ nos permite controlar la longitud de paso para abordar los problemas de convergencia.

## Aplicación
Implementar Adam desde cero no es muy desalentador. Para mayor comodidad almacenamos el contador de paso de tiempo $t$ en el diccionario `hyperparams`. Más allá de eso todo es sencillo.


In [ ]:
%matplotlib inline
import torch
from laboratorio import d2l


def init_adam_states(feature_dim):
    v_w, v_b = torch.zeros((feature_dim, 1)), torch.zeros(1)
    s_w, s_b = torch.zeros((feature_dim, 1)), torch.zeros(1)
    return ((v_w, s_w), (v_b, s_b))

def adam(params, states, hyperparams):
    beta1, beta2, eps = 0.9, 0.999, 1e-6
    for p, (v, s) in zip(params, states):
        with torch.no_grad():
            v[:] = beta1 * v + (1 - beta1) * p.grad
            s[:] = beta2 * s + (1 - beta2) * torch.square(p.grad)
            v_bias_corr = v / (1 - beta1 ** hyperparams['t'])
            s_bias_corr = s / (1 - beta2 ** hyperparams['t'])
            p[:] -= hyperparams['lr'] * v_bias_corr / (torch.sqrt(s_bias_corr)
                                                       + eps)
        p.grad.data.zero_()
    hyperparams['t'] += 1

Estamos listos para utilizar Adam para entrenar el modelo. Utilizamos una tasa de aprendizaje de $\eta = 0.01$.


In [ ]:
data_iter, feature_dim = d2l.get_data_ch11(batch_size=10)
d2l.train_ch11(adam, init_adam_states(feature_dim),
               {'lr': 0.01, 't': 1}, data_iter, feature_dim);

Una implementación más concisa es sencilla ya que `adam` es uno de los algoritmos proporcionados como parte de la biblioteca de optimización Gluon `trainer`. Por lo tanto, solo necesitamos pasar parámetros de configuración para una implementación en Gluon.


### Nota docente de Hespérides

Momentum mantiene una media del gradiente; Adam combina estimaciones de primer y segundo momento con corrección del sesgo inicial. Su división por una escala por coordenada cambia la geometría efectiva del paso. No elimina la elección de tasa ni garantiza mejor generalización. Usa el explorador 90 para comparar trayectorias y pérdida desde un mismo punto; después distingue ese experimento determinista del ruido introducido por minibatches.

Vínculo con los apuntes: sesión 2, «Adam y comparación controlada con SGD».


In [ ]:
trainer = torch.optim.Adam
d2l.train_concise_ch11(trainer, {'lr': 0.01}, data_iter)

## Yogi
Uno de los problemas de Adam es que no puede converger incluso en los ajustes convexos cuando el segundo momento estimado en $\mathbf{s}_t$ explota. Como una solución [Zaheer.Reddi.Sachan.ea.2018](https://d2l.ai/chapter_references/zreferences.html) propuso una actualización refinada (y la inicialización) para $\mathbf{s}_t$. Para entender lo que está pasando, vamos a reescribir la actualización de Adam como sigue:

$$\mathbf{s}_t \leftarrow \mathbf{s}_{t-1} + (1 - \beta_2) \left(\mathbf{g}_t^2 - \mathbf{s}_{t-1}\right).$$

Cada vez que $\mathbf{g}_t^2$ tiene alta varianza o actualizaciones son escasas, $\mathbf{s}_t$ puede olvidar valores pasados demasiado rápido. Una posible solución para esto es reemplazar $\mathbf{g}_t^2 - \mathbf{s}_{t-1}$ por $\mathbf{g}_t^2 \odot \mathop{\textrm{sgn}}(\mathbf{g}_t^2 - \mathbf{s}_{t-1})$. Ahora la magnitud de la actualización ya no depende de la cantidad de desviación. Esto produce las actualizaciones Yogi

$$\mathbf{s}_t \leftarrow \mathbf{s}_{t-1} + (1 - \beta_2) \mathbf{g}_t^2 \odot \mathop{\textrm{sgn}}(\mathbf{g}_t^2 - \mathbf{s}_{t-1}).$$

Además, los autores aconsejan inicializar el impulso en un lote inicial más grande que sólo una estimación puntual inicial. Omitimos los detalles ya que no son materiales para la discusión y ya que incluso sin esta convergencia sigue siendo bastante bueno.


In [ ]:
def yogi(params, states, hyperparams):
    beta1, beta2, eps = 0.9, 0.999, 1e-3
    for p, (v, s) in zip(params, states):
        with torch.no_grad():
            v[:] = beta1 * v + (1 - beta1) * p.grad
            s[:] = s + (1 - beta2) * torch.sign(
                torch.square(p.grad) - s) * torch.square(p.grad)
            v_bias_corr = v / (1 - beta1 ** hyperparams['t'])
            s_bias_corr = s / (1 - beta2 ** hyperparams['t'])
            p[:] -= hyperparams['lr'] * v_bias_corr / (torch.sqrt(s_bias_corr)
                                                       + eps)
        p.grad.data.zero_()
    hyperparams['t'] += 1

data_iter, feature_dim = d2l.get_data_ch11(batch_size=10)
d2l.train_ch11(yogi, init_adam_states(feature_dim),
               {'lr': 0.01, 't': 1}, data_iter, feature_dim);

## Resumen
* Adam combina características de muchos algoritmos de optimización en una regla de actualización bastante robusta.
* Creado sobre la base de RMSProp, Adam también utiliza EWMA en el gradiente estocástico minibatch.
* Adam utiliza la corrección de sesgo para ajustar para un inicio lento al estimar el momento y un segundo momento.
* Para los gradientes con varianza significativa podemos encontrar problemas con la convergencia. Pueden ser modificados mediante el uso de minibatches más grandes o cambiando a una estimación mejorada para $\mathbf{s}_t$. Yogi ofrece tal alternativa.

## Ejercicios
1. Ajustar la tasa de aprendizaje y observar y analizar los resultados experimentales.
1. ¿Se puede reescribir el momento y las actualizaciones de segundo momento de tal manera que no requiere corrección de sesgo?
1. ¿Por qué necesitas reducir la tasa de aprendizaje $\eta$ a medida que confluimos?
1. ¿Intentar construir un caso para el que Adam diverge y Yogi converge?


[Debate del original](https://discuss.d2l.ai/t/1078)
